# param-grad-access — ex1: iterate model.parameters() and read .grad with the None guard

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `param-grad-access`. Running the final beacon cell reports progress against the `PyTorch: param.grad access` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: param.grad access` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`param-grad-access`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "param-grad-access"
DD_SUBTOPIC = "PyTorch: param.grad access"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## param.grad access — quick refresher

In PyTorch (and any clone of its API), a `Parameter` is a `Tensor` subclass that *participates in backprop*. After `loss.backward()`, each parameter holds its accumulated gradient on the `.grad` attribute, matching the parameter's shape:

```python
for p in model.parameters():
    print(p.shape, p.grad.shape)   # always equal
    p.data -= lr * p.grad           # SGD step
    p.grad = None                   # zero_grad: drop the tensor
```

- `p.grad is None` BEFORE any backward pass — guard against this.
- `p.grad` is **accumulated**: every backward pass *adds* to it (so you must zero it between steps).
- The standard zero strategy is `p.grad = None`, not `p.grad.zero_()` — saves memory and matches PyTorch's default in `optimizer.zero_grad(set_to_none=True)`.

### Exercise 1 — iterate model.parameters() and read .grad with the None guard

> ```yaml
> Difficulty: 🔴🔴⚪⚪⚪
> Bloom level: Apply
> LO: Apply the parameters() → p.grad pattern (with None-guard) to compute an SGD step manually across a multi-parameter module.
> Keywords: parameters, grad, iterate, none-guard, sgd-step
> ```

**KCs targeted:** `param-grad-access`, `zero-grad-set-none`

Implement `manual_sgd_step(params, lr)`. Given an iterable of tensors that already have `.grad` populated (or `.grad is None`), apply the canonical SGD step to each.

**For each parameter p:**
1. If `p.grad is None`, **skip** it (the standard guard — a parameter    that didn't participate in any forward pass has no gradient yet,    and you can't subtract `None` from a tensor).
2. Otherwise update **in place**: `p.data -= lr * p.grad`. We use    `.data` to bypass autograd tracking (we don't want the SGD step    itself to build a Recipe).

**Then implement `zero_grads(params)`** — for each parameter, set `p.grad = None`. This is the PyTorch-recommended zero strategy (memory-cheaper than `p.grad.zero_()` because the gradient tensor becomes garbage-collectable, and matches `optimizer.zero_grad(set_to_none=True)`).

Inputs are plain `nn.Parameter` instances (a Tensor subclass). The test populates `.grad` by hand (no real backward pass needed) and checks that updates match `p_new = p_old - lr * grad` elementwise.

In [ ]:
def manual_sgd_step(params, lr: float) -> None:
    for p in params:
        if p.grad is None:
            # Param didn't see any forward pass since last zero_grad.
            # Subtracting None from a tensor raises TypeError, so skip.
            continue
        # In-place: .data bypasses autograd tracking for the update itself.
        p.data -= lr * p.grad


def zero_grads(params) -> None:
    for p in params:
        # set_to_none=True style — cheaper than p.grad.zero_().
        # The grad tensor becomes garbage-collectable.
        p.grad = None


<details><summary>Solution</summary>

```python
def manual_sgd_step(params, lr: float) -> None:
    for p in params:
        if p.grad is None:
            # Param didn't see any forward pass since last zero_grad.
            # Subtracting None from a tensor raises TypeError, so skip.
            continue
        # In-place: .data bypasses autograd tracking for the update itself.
        p.data -= lr * p.grad


def zero_grads(params) -> None:
    for p in params:
        # set_to_none=True style — cheaper than p.grad.zero_().
        # The grad tensor becomes garbage-collectable.
        p.grad = None
```

**Why guard on `p.grad is None`.** In PyTorch, fresh `Parameter`s have `.grad = None` until the first `backward()` populates them. After `zero_grad(set_to_none=True)`, `.grad` becomes `None` again. If you blindly do `p.data -= lr * p.grad` you'll get `TypeError: unsupported operand type(s) for *: 'float' and 'NoneType'`.

**Why `p.data -=` and not `p -=`.** `p -= lr * p.grad` would trigger autograd because `p.requires_grad` is True (Parameters always do). `.data` is the raw underlying tensor — writing to it doesn't build a Recipe. The same reason ARENA wraps the SGD step in `NoGrad()`.

**Why `p.grad = None` over `p.grad.zero_()`.** Two reasons:
- Memory: the gradient tensor becomes garbage-collectable, freeing up   GPU memory between forward passes. `zero_()` keeps the allocation   alive.
- Semantics: `None` means "no gradient computed yet" (a clearer   invariant than "the zero tensor pretending to be no gradient").   This is why PyTorch made `set_to_none=True` the default in 1.7.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()